# 01 — Data Preprocessing


In [1]:
# ---- Shared configuration -------------------------------------------------
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

DATA_PATH = "/kaggle/input/datasets/fahmidaalamtamim/final-pcf-dataset-last-one/FINAL_PCF_DATASET_cleaned.xlsx"   # <-- adjust (e.g. /kaggle/input/.../FINAL_PCF_DATASET.xlsx)
OUT_DIR = Path("outputs_pcf_xai"); OUT_DIR.mkdir(exist_ok=True)
PROCESSED = OUT_DIR / "processed_dataset.npz"

RESPONSES = ["real ewfd", "imag ewfd", "Aeff", "dispersion(y)",
             "non linearity(y)", "conf loss(y)"]
TARGETS = ["Ratio", "Shape", "No. of rings", "Anomaly position"]

N_AUG = 40; NOISE_FRAC = 0.02; N_FOLDS = 5
EPOCHS = 300; PATIENCE = 30; LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cpu


## 1.1 Raw data inspection

In [2]:
df_raw = pd.read_excel(DATA_PATH)
print(df_raw.shape); df_raw.head()

(5141, 11)


,Ratio,Shape,No. of rings,Anomaly position,Wavelength,real ewfd,imag ewfd,Aeff,dispersion(y),non linearity(y),conf loss(y)
0,0.7,6,7,2,0.64,2.8659,1.425400e-17,2.852900e-12,-10666.666667,9.291335,-1.215507e-11
1,0.7,6,7,2,0.65,2.8560,3.770600e-18,2.721200e-12,-10833.333333,9.591153,-3.165905e-12
2,0.7,6,7,2,0.66,2.8466,3.770600e-18,2.725100e-12,-8800.000000,9.432314,-3.117936e-12
3,0.7,6,7,2,0.67,2.8376,3.770600e-18,2.729000e-12,-8933.333333,9.278255,-3.071400e-12
4,0.7,6,7,2,0.68,2.8292,1.060900e-19,2.729000e-12,-9066.666667,9.141810,-8.514639e-14


In [3]:
# unique structures and label balance
uniq = df_raw[TARGETS].drop_duplicates()
print("unique structures:", len(uniq))
for t in TARGETS:
    print(t, dict(uniq[t].value_counts()))

unique structures: 53
Ratio {0.7: np.int64(27), 0.65: np.int64(26)}
Shape {6: np.int64(18), 7: np.int64(18), 8: np.int64(17)}
No. of rings {8: np.int64(18), 6: np.int64(18), 7: np.int64(17)}
Anomaly position {2: np.int64(18), 3: np.int64(18), 4: np.int64(17)}


## 1.2 Structure-level table
One structure was simulated twice; the duplicate sweeps are averaged. Dispersion,
confinement loss and Im(n_eff) span many orders of magnitude (dispersion up to
~1e7, conf loss ~1e-11), so a signed `log1p` transform (median-scaled) is applied
before standardisation.

In [4]:
def load_structure_table():
    df = pd.read_excel(DATA_PATH)
    # average duplicated sweeps (one structure was simulated twice)
    df = (df.groupby(TARGETS + ["Wavelength"], as_index=False)[RESPONSES].mean())

    wavelengths = np.sort(df["Wavelength"].unique())
    # signed log transform for heavy-tailed responses (dispersion spans 1e7)
    for col in ["dispersion(y)", "conf loss(y)", "imag ewfd"]:
        df[col] = np.sign(df[col]) * np.log1p(np.abs(df[col]) /
                                              (np.abs(df[col]).median() + 1e-30))

    rows, labels = [], []
    for key, g in df.groupby(TARGETS):
        g = g.sort_values("Wavelength")
        assert len(g) == len(wavelengths), f"incomplete sweep for {key}"
        feat = np.concatenate([g[r].to_numpy() for r in RESPONSES])
        rows.append(feat)
        labels.append(key)

    X = np.vstack(rows)                                   # (53, 528)
    y_raw = pd.DataFrame(labels, columns=TARGETS)
    # encode each target to 0..K-1
    y, classes = {}, {}
    for t in TARGETS:
        cats = np.sort(y_raw[t].unique())
        classes[t] = cats
        y[t] = y_raw[t].map({c: i for i, c in enumerate(cats)}).to_numpy()

    feat_names = [f"{r}@{w:.2f}" for r in RESPONSES for w in wavelengths]
    return X, y, classes, wavelengths, feat_names


In [5]:
X, y, classes, wavelengths, feat_names = load_structure_table()
n_wl = len(wavelengths)
print(f"X: {X.shape}  ({len(RESPONSES)} responses x {n_wl} wavelengths)")
for t in TARGETS:
    print(t, "classes:", classes[t])

X: (53, 582)  (6 responses x 97 wavelengths)
Ratio classes: [0.65 0.7 ]
Shape classes: [6 7 8]
No. of rings classes: [6 7 8]
Anomaly position classes: [2 3 4]


In [6]:
stats = pd.DataFrame({
    "n structures": [X.shape[0]],
    "features (full)": [X.shape[1]],
    "responses": [len(RESPONSES)],
    "wavelength points": [n_wl],
    "wavelength range (um)": [f"{wavelengths.min():.2f}-{wavelengths.max():.2f}"],
}).T.rename(columns={0: "value"})
stats

,value
n structures,53
features (full),582
responses,6
wavelength points,97
wavelength range (um),0.64-1.60


## 1.4 Save processed dataset

In [7]:
save = {"X": X, "wavelengths": wavelengths,
        "feat_names": np.array(feat_names)}
for i, t in enumerate(TARGETS):
    save[f"y_{i}"] = y[t]
    save[f"c_{i}"] = np.asarray(classes[t])
np.savez_compressed(PROCESSED, **save)
print("saved ->", PROCESSED)

saved -> outputs_pcf_xai/processed_dataset.npz


In [8]:
import os

print("Current directory:", os.getcwd())

print("\nFiles in /kaggle/working:")
print(os.listdir("/kaggle/working"))

Current directory: /kaggle/working

Files in /kaggle/working:
['outputs_pcf_xai', '.virtual_documents']


In [9]:
print("OUT_DIR =", OUT_DIR)
print("PROCESSED =", PROCESSED)

OUT_DIR = outputs_pcf_xai
PROCESSED = outputs_pcf_xai/processed_dataset.npz


In [10]:
import glob

print(glob.glob("/kaggle/working/**/*", recursive=True))

['/kaggle/working/outputs_pcf_xai', '/kaggle/working/outputs_pcf_xai/processed_dataset.npz']
